# Phase 0 walkthrough — the frontier pipeline, one stage at a time

`run_frontier.py` does all of this in one shot and prints a table. This notebook
opens up the middle so you can see what each stage is actually holding.

| Stage | Question it answers |
|---|---|
| 1 Catalog | What does a real tool look like, and how is it identified? |
| 2 Examples | What is ground truth here, and what gets thrown away? |
| 3 Budget | What does "context bloat" cost in actual tokens? |
| 4 Split | Where was the leakage, and what did fixing it cost? |
| 5 One decision | What does each baseline *pick* for a single query? |
| 6 Sweep | The full budget × selector grid |
| 7 Frontier | The chart |
| 8 Caveats | Why none of this is the gate yet |

Run it with:

```sh
uv run --extra dev --extra bench --with jupyter jupyter lab notebooks/frontier_walkthrough.ipynb
```

In [1]:
import json
from collections import Counter
from pathlib import Path

from mcp_gateway_router.baselines import (
    ExposeAll, LexicalScorer, Oracle, PopularityTopK,
    ProgressiveDisclosure, SemanticRetrieval, StaticSet,
)
from mcp_gateway_router.data import load_catalog, load_examples, restrict_to_catalog
from mcp_gateway_router.frontier import evaluate, sweep
from mcp_gateway_router.run_frontier import split_examples
from mcp_gateway_router.tokens import StaticTokenCounter, total_cost

TOOLS_SUBSET, QUERIES_SUBSET = "web", "apibank"
BUDGETS = [500, 1000, 2000, 4000, 8000, 16000, 32000]
STATIC_SIZE = 25

# Flat cost stands in for measured cost. Stage 3 shows why that matters.
counter = StaticTokenCounter({}, default=120)

## Stage 1 — The catalog

ToolRet's `web` subset. The gold tools for our queries live in here alongside
37k others, which is the point: a realistic haystack, not a curated shortlist.

Look at `uid` — identity is `(server_id, name)` plus a schema content hash, so a
vendor renaming a field invalidates that one tool and nothing else.

In [2]:
catalog = load_catalog(TOOLS_SUBSET)
print(f"{len(catalog):,} tools")

servers = Counter(t.server_id for t in catalog)
print("\ntop source servers:")
for name, n in servers.most_common(8):
    print(f"  {name:<24}{n:>7,}")

sample = catalog.get("apibank", "apibank_tool_33")
print(f"\n--- one tool ---\nkey:  {sample.key}\nuid:  {sample.uid}")
print(f"schema: {json.dumps(sample.input_schema)[:160]}")
print(f"description:\n{sample.description[:400]}")

37,292 tools

top source servers:
  toolACE                  16,072
  toolbench                13,862
  apigen                    3,605
  ultraTool                 1,171
  rotbench                    919
  reversechain                783
  toolLens                    314
  apibank                     101

--- one tool ---
key:  ('apibank', 'apibank_tool_33')
uid:  apibank/apibank_tool_33@a67cfeddd0818165
schema: {"type": "object", "properties": {"args": [{"name": "date", "in": "query", "schema": {"type": "str", "default": null}, "description": "The date for which histor
description:
"QueryHistoryToday API takes one parameter 'date'. The parameter 'date' refers to the date for which historical events are to be queried."


## Stage 2 — Ground truth, and what gets discarded

Each example is a query plus the tools it actually needed. This is the
full-information holdout that makes exact policy value computable without an
estimator — and the reason this harness validates machinery rather than value.

`restrict_to_catalog` drops queries whose gold tools aren't in the loaded subset.
Keeping them would make the oracle unreachable and break the frontier's reference line.

In [3]:
raw = load_examples(QUERIES_SUBSET, limit=200)
examples = restrict_to_catalog(raw, catalog)
print(f"loaded {len(raw)} queries -> {len(examples)} usable ({len(raw) - len(examples)} dropped)")

for e in examples[:3]:
    print(f"\nQ: {e.task}")
    for key in e.required:
        print(f"   needs {key[1]}")

sizes = Counter(len(e.required) for e in examples)
print(f"\ntools required per query: {dict(sorted(sizes.items()))}")

loaded 101 queries -> 101 usable (0 dropped)

Q: Can you tell me about the historical events of April 21st?
   needs apibank_tool_33
   needs apibank_tool_53

Q: Can you tell me today's date?
   needs apibank_tool_23

Q: Can you tell me the current date?
   needs apibank_tool_23

tools required per query: {1: 48, 2: 38, 3: 12, 4: 3}


## Stage 3 — The budget, in tokens

This is the whole premise made concrete. Schemas sit in the prompt prefix on
**every turn**, so this cost is paid repeatedly — and on the customer's bill.

The flat 120/tool below is a placeholder. Real measurement uses
`AnthropicTokenCounter`, which calls `messages.count_tokens` with the tool in the
`tools` array. That matters because the knapsack is only interesting when schema
sizes actually differ — a flat cost silently turns it back into a top-K cut.

In [4]:
full = total_cost(counter, catalog)
print(f"whole catalog: {full:,} tokens  (~{full / 1_000_000:.1f}x a 1M context window)")
print(f"a 25-tool set: {25 * 120:,} tokens")
print(f"fraction of the catalog you can afford at 25 tools: {25 / len(catalog):.3%}")

# What a *real* counter would show: schema sizes are wildly uneven.
raw_sizes = sorted(len(json.dumps(t.input_schema)) for t in catalog)
mid, p95 = raw_sizes[len(raw_sizes) // 2], raw_sizes[int(len(raw_sizes) * 0.95)]
print(f"\nserialised schema chars — median {mid}, p95 {p95}, max {raw_sizes[-1]:,}")
print("^ this spread is what makes it a knapsack rather than a top-K cut,")
print("  and it is exactly what the flat placeholder erases.")

whole catalog: 4,475,040 tokens  (~4.5x a 1M context window)


a 25-tool set: 3,000 tokens
fraction of the catalog you can afford at 25 tools: 0.067%

serialised schema chars — median 178, p95 795, max 15,174
^ this spread is what makes it a knapsack rather than a top-K cut,
  and it is exactly what the flat placeholder erases.


## Stage 4 — The fit/eval split

`static-set` and `popularity` are *history-dependent*: they need past usage to pick
a set. The first run fitted them on the same examples they were scored on, which
with 101 tools and 101 queries meant they were reading the answer key.

The cell below shows the overlap that leaked. The other four baselines learn
nothing, so they never touch the fit half.

In [5]:
fit, ev = split_examples(examples)
fit_gold = Counter(k for e in fit for k in e.required)
eval_gold = {k for e in ev for k in e.required}

print(f"fit={len(fit)}  eval={len(ev)}")
print(f"distinct gold tools — fit {len(fit_gold)}, eval {len(eval_gold)}")

static_keys = [k for k, _ in fit_gold.most_common(STATIC_SIZE)]
covered = sum(1 for k in static_keys if k in eval_gold)
print(f"\nthe hand-picked {len(static_keys)} covers {covered} of {len(eval_gold)} eval-needed tools")
print("Before the split this was ~100% by construction — that was the leak.")

fit=50  eval=51
distinct gold tools — fit 39, eval 32

the hand-picked 25 covers 18 of 32 eval-needed tools
Before the split this was ~100% by construction — that was the leak.


## Stage 5 — One decision, watched

The most useful cell in the notebook. For a single query at a single budget, what
does each baseline actually put in front of the model?

Note what `semantic-retrieval` returns versus what was needed — the gap between
"looks topically related" and "is the tool that does the job" is the whole problem.

In [6]:
scorer = LexicalScorer(catalog)  # placeholder — see Stage 8
selectors = [
    ExposeAll(),
    StaticSet(static_keys),
    PopularityTopK(dict(fit_gold)),
    SemanticRetrieval(scorer),
    ProgressiveDisclosure(scorer, core_keys=static_keys[:3]),
    Oracle({e.task: list(e.required) for e in ev}),
]

probe, BUDGET = ev[0], 2000
print(f"Q: {probe.task}\nneeds: {[k[1] for k in probe.required]}\nbudget: {BUDGET} tokens\n")

for s in selectors:
    picked = s.select(probe.decision_context(), catalog, BUDGET, counter)
    keys = {t.key for t in picked}
    hit = set(probe.required) <= keys
    head = ", ".join(t.name for t in picked[:4]) or "(nothing)"
    flag = "HIT " if hit else "MISS"
    print(f"{flag} {s.name:<24}{len(picked):>6} tools  {total_cost(counter, picked):>8,} tok")
    print(f"     {head}{' ...' if len(picked) > 4 else ''}")

Q: Can you please tell me the current date?
needs: ['apibank_tool_23']
budget: 2000 tokens

HIT  expose-all               37292 tools  4,475,040 tok
     ultraTool_tool_0, ultraTool_tool_1, ultraTool_tool_2, ultraTool_tool_3 ...
HIT  static-set                  16 tools     1,920 tok
     apibank_tool_0, apibank_tool_61, apibank_tool_62, apibank_tool_23 ...


HIT  popularity                  16 tools     1,920 tok
     apibank_tool_0, apibank_tool_61, apibank_tool_2, apibank_tool_23 ...
MISS semantic-retrieval          16 tools     1,920 tok
     toolbench_tool_12487, toolLens_tool_55, toolbench_tool_9137, tooleyes_tool_5 ...
MISS progressive-disclosure       4 tools       480 tok
     find_tools, apibank_tool_0, apibank_tool_61, apibank_tool_62
HIT  oracle                       1 tools       120 tok
     apibank_tool_23


## Stage 6 — The sweep

Every selector at every budget. `mean_round_trips` is non-zero only for progressive
disclosure: recovery is no longer free, because deferring a schema is not the same
as avoiding it.

In [7]:
points = sweep(selectors, ev, catalog, BUDGETS, counter)

try:
    import pandas as pd
    df = pd.DataFrame([p.__dict__ for p in points])
    display(df[["selector", "budget", "mean_tokens", "satisfied", "recall",
                "mean_round_trips", "exposure_entropy"]].round(3))
except ImportError:
    for p in points:
        print(f"{p.selector:<24}{p.budget:>7}{p.mean_tokens:>10,.0f}{p.satisfied:>9.3f}")

,selector,budget,mean_tokens,satisfied,recall,mean_round_trips,exposure_entropy
0,expose-all,500,4475040.000,1.000,1.000,0.00,15.187
1,expose-all,1000,4475040.000,1.000,1.000,0.00,15.187
2,expose-all,2000,4475040.000,1.000,1.000,0.00,15.187
3,expose-all,4000,4475040.000,1.000,1.000,0.00,15.187
4,expose-all,8000,4475040.000,1.000,1.000,0.00,15.187
5,expose-all,16000,4475040.000,1.000,1.000,0.00,15.187
6,expose-all,32000,4475040.000,1.000,1.000,0.00,15.187
7,static-set,500,480.000,0.039,0.211,0.00,2.000
8,static-set,1000,960.000,0.078,0.301,0.00,3.000
9,static-set,2000,1920.000,0.412,0.547,0.00,4.000


## Stage 7 — The frontier

X is tokens *actually spent*, not the budget knob — `expose-all` ignores the budget,
so plotting against the knob would put it at every X while it only ever spends one
amount. Dominating means sitting up and to the left.

In [8]:
import html as _html
import warnings

from IPython.display import HTML, FileLink, display

from mcp_gateway_router.report import render

payload = {
    "meta": {
        "tools_subset": TOOLS_SUBSET, "queries_subset": QUERIES_SUBSET,
        "catalog_size": len(catalog), "n_fit": len(fit), "n_examples": len(ev),
        "budgets": BUDGETS, "scorer": "lexical (PLACEHOLDER)",
        "token_costs": "flat estimate",
    },
    "points": [p.__dict__ for p in points],
}
doc = render(payload)

out = Path("../results/frontier_notebook.html")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(doc)

# Embed via srcdoc rather than src. A relative `src` resolves against the *notebook's
# URL*, not the filesystem — under JupyterLab `../results/...` lands on an app route,
# and VS Code's sandboxed output webview blocks local relative paths outright. Either
# way you get an empty box and no error, because IFrame never validates its src.
# srcdoc carries the document inline, so there is no path left to resolve.
#
# IPython warns "consider using IFrame instead" for raw iframe HTML — that is exactly
# the advice that produces the empty box here, so it is suppressed deliberately.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*IPython.display.IFrame.*")
    display(HTML(
        f'<iframe srcdoc="{_html.escape(doc, quote=True)}" '
        'width="100%" height="780" style="border:1px solid #33332f;border-radius:8px"></iframe>'
    ))

display(FileLink(str(out), result_html_prefix="Standalone (full interactivity): "))

/Users/nguyenvietkhoi/mcp-gateway-router/results/frontier_notebook.html

## Stage 8 — What's fixed, and what still blocks the gate

**Fixed since the first run:**

1. **Progressive disclosure can now lose.** It used to score 1.000 at every budget
   because recovery was free and always succeeded — a benchmark that couldn't fail.
   `find_tools` now runs *the same retriever baseline 4 gets*, returns `k` schemas, and
   the agent pays for all of them plus a round trip; recovery succeeds only if the search
   actually found the tool. On the real run this moved PD from **1.000 → 0.392**, closely
   matching the 70-point overstatement this notebook measured. The cell below verifies it
   directly, by showing PD failing at small `k`.
2. **Baseline 4 has a real bi-encoder.** `--embeddings` uses
   `mangopy/ToolRet-trained-e5-base-v2`. It helps most at small budgets
   (0.099 → 0.255 satisfied at 500 tokens), which is exactly where the frontier is
   decided. This notebook still runs `LexicalScorer` for speed — the numbers above are
   the lexical ones.
3. **Leakage** — `static-set` and `popularity` now fit on a disjoint half (Stage 4).

**Still open:**

- **Token costs are flat (120/tool).** Real spread is ~85× (Stage 3). Needs
  `--count-tokens` and Anthropic credentials. Until then the knapsack is a top-K cut in
  disguise.
- **There is no user in this data.** No tenant, seat, session, or history — so this
  framework *cannot evaluate personalization at all*, only retrieval. That is Phase 1's
  job, and even Phase 1 is synthetic because we design the personas.
- **Catalog composition is unresolved.** `web` merges eight corpora; our gold tools are
  0.27% of it. That matches "a proxy over a fleet of servers", not "one vendor, 400 tools,
  one domain" — and retrieval looks far worse in the former. **Which setting the gate is
  about is a decision, not a measurement.**


In [ ]:
# Verify the fix: progressive disclosure must be able to LOSE.
# Sweep k — the number of schemas find_tools returns. Small k means the search
# genuinely misses, and PD fails the task while still paying for the round trip.
print(f"{'k':>4}{'satisfied':>12}{'recovered':>12}{'round trips':>14}{'tokens':>10}")
for k in (1, 3, 10, 50):
    pd_k = ProgressiveDisclosure(scorer, core_keys=static_keys[:3], k=k)
    pt = evaluate(pd_k, ev, catalog, 2000, counter)
    print(f"{k:>4}{pt.satisfied:>12.3f}{pt.recovered:>12.3f}"
          f"{pt.mean_round_trips:>14.2f}{pt.mean_tokens:>10.0f}")

print("\nsatisfied < 1.0 anywhere means the baseline is falsifiable — that is the fix.")
print("Note tokens rising with k: you pay for every schema the search returns,")
print("not just the one you needed. PD defers cost, it does not avoid it.")